# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR^2 colorectal cancer survivors dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p) using the `mlcroissant` library. All dataset elements (record sets, fields, columns, etc.) are referenced strictly by their `@id` values to ensure consistency.

### Dataset Source
The dataset schema is published as Croissant JSON-LD at:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset's metadata and explore its key descriptive information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset object
dataset = mlc.Dataset(croissant_url)

meta = dataset.metadata
print(f"Dataset name: {meta.name}")
print(f"Description: {meta.description}")
print(f"Identifier: {meta.identifier}")
print(f"License: {meta.license}")
print(f"Number of record sets: {len(meta.record_sets)}")

## 2. Data Overview
Let's review the available record sets and their fields, referencing all by their `@id`s.

We will print the `@id`s and names for each record set and, within each, the fields and columns, if any.

In [ ]:
# Explore record sets and their fields by @id

from pprint import pprint

record_sets = meta.record_sets
print(f"Found {len(record_sets)} record sets:")

for rs in record_sets:
    print(f"\nRecord set @id: {rs['@id']} | Name: {rs.get('name', 'N/A')}")
    if 'fields' in rs:
        for f in rs['fields']:
            print(f"  Field @id: {f['@id']} | Name: {f.get('name', 'N/A')} | Data type: {f.get('dataType', 'N/A')}")
            if 'column' in f:
                col = f['column']
                print(f"    Column @id: {col.get('@id', 'N/A')}")
    else:
        print("  No fields defined.")

## 3. Data Extraction
Now, we will extract tabular data from each record set using their `@id` and load it into pandas DataFrames. 

**Note:** All interactions are performed via `@id` to ensure reproducibility and clarity.

> We will list all record set `@id`s, and for demonstration, load and display the first record set.

In [ ]:
# Gather all record set @id values
rs_ids = [rs['@id'] for rs in record_sets]
print("Record sets (by @id):")
for rs_id in rs_ids:
    print(f"  - {rs_id}")

# Load each record set into a DataFrame
dataframes = {}
for rs_id in rs_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
    else:
        print(f"No records found for record set {rs_id}.")

# Choose first record set for display
if dataframes:
    first_rs_id = list(dataframes)[0]
    print(f"\nColumns for {first_rs_id}:\n{dataframes[first_rs_id].columns.tolist()}")
    display(dataframes[first_rs_id].head())
else:
    print("No tabular data available in record sets.")

## 4. Exploratory Data Analysis (EDA)
Here we apply simple data processing steps, such as filtering, normalization, and grouping using the DataFrame columns. Make sure to reference all fields by their `@id` as reflected in the dataset.

Let's illustrate with a numeric field present in the first record set (for example, "age at 2nd diagnosis" if available). If no numeric field is present, you may need to adapt the code accordingly for your analysis.

In [ ]:
# Example EDA: Use field @ids to select numeric field for filtering/normalization

# Show available columns in the first record set for guidance
if dataframes:
    first_df = dataframes[first_rs_id]
    print(f"Available columns in {first_rs_id}:\n{first_df.columns.tolist()}")

    # Let's try to pick out a likely numeric field by iterating columns
    # If not found, adapt this cell to match an actual numeric field @id from above
    import numpy as np
    numeric_cols = first_df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        numeric_field = numeric_cols[0]
        print(f"Using numeric field '@id': {numeric_field}")
    else:
        print("No numeric fields detected in first record set. Please adapt to your data.")
        numeric_field = None
else:
    numeric_field = None

# Choose an arbitrary threshold and filter (if a numeric_field is available)
if numeric_field:
    threshold = first_df[numeric_field].quantile(0.95)  # Top 5% as example outlier threshold
    filtered_df = first_df[first_df[numeric_field] < threshold]
    print(f"Filtered records with {numeric_field} < {threshold:.2f} (removes outliers):")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field}:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by another available field if possible
    # Try to pick a categorical/text field
    possible_groups = first_df.select_dtypes(include=['object']).columns.tolist()
    # Exclude fields that are only unique IDs
    group_field = None
    for col in possible_groups:
        if first_df[col].nunique() < len(first_df) // 2:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped average {numeric_field} by '{group_field}':")
        display(grouped_df.head())
    else:
        print("No suitable categorical field to group by.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field and its normalization, as well as an example categorical grouping if available.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if numeric_field and not filtered_df.empty:
    fig, ax = plt.subplots(1,2, figsize=(12,5))
    filtered_df[numeric_field].hist(ax=ax[0], bins=10, color='skyblue', edgecolor='k')
    ax[0].set_title(f"Distribution of {numeric_field}")
    if f"{numeric_field}_normalized" in filtered_df.columns:
        filtered_df[f"{numeric_field}_normalized"].hist(ax=ax[1], bins=10, color='salmon', edgecolor='k')
        ax[1].set_title(f"Normalized {numeric_field}")
    plt.show()

    # Boxplot by group_field if found
    if 'group_field' in locals() and group_field:
        filtered_df.boxplot(column=numeric_field, by=group_field, rot=60, grid=False, fontsize=10)
        plt.title(f"{numeric_field} by {group_field}")
        plt.suptitle("")
        plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load and explore the **Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors** dataset, referencing all key dataset elements by their stable `@id` identifiers.

- We programmatically inspected metadata, listed all record sets, and loaded tabular data strictly via their declared Croissant `@id`s.
- Simple EDA showed how to filter outliers, normalize fields, and perform basic grouping — all guided by metadata.

This template workflow can be extended to richer transformations or analyses, always referencing dataset elements by `@id` for complete reproducibility.